# Captum TCAV Stripes Proof-of-Concept

This notebook reproduces a stripes-based TCAV concept interpretability workflow using a synthetic dataset and a small CNN.


In [ ]:
# imports
import random

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from captum.concept import TCAV, Concept

from captum.concept._utils.classifier import DefaultClassifier

# If needed, uncomment:
# %pip install torch torchvision captum numpy scikit-learn matplotlib pillow

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)


In [ ]:
# dataset generation
IMG_SIZE = 32

def _normalize01(x):
    return np.clip(x, 0.0, 1.0).astype(np.float32)

def make_horizontal_stripes(size=IMG_SIZE, stripe_width=4, noise=0.08):
    img = np.zeros((size, size), dtype=np.float32)
    for r in range(size):
        img[r, :] = 1.0 if ((r // stripe_width) % 2 == 0) else 0.0
    img += np.random.normal(0.0, noise, size=(size, size)).astype(np.float32)
    return _normalize01(img)

def make_vertical_stripes(size=IMG_SIZE, stripe_width=4, noise=0.08):
    img = np.zeros((size, size), dtype=np.float32)
    for c in range(size):
        img[:, c] = 1.0 if ((c // stripe_width) % 2 == 0) else 0.0
    img += np.random.normal(0.0, noise, size=(size, size)).astype(np.float32)
    return _normalize01(img)

def make_random_noise(size=IMG_SIZE):
    return np.random.rand(size, size).astype(np.float32)

def to_3ch_tensor(img2d):
    t = torch.from_numpy(img2d).unsqueeze(0)  # 1xHxW
    t = t.repeat(3, 1, 1)  # 3xHxW
    return t

class StripeClassificationDataset(Dataset):
    # label: 0 = horizontal, 1 = vertical
    def __init__(self, n_per_class=500):
        images = []
        labels = []
        for _ in range(n_per_class):
            images.append(to_3ch_tensor(make_horizontal_stripes()))
            labels.append(0)
        for _ in range(n_per_class):
            images.append(to_3ch_tensor(make_vertical_stripes()))
            labels.append(1)

        self.x = torch.stack(images)
        self.y = torch.tensor(labels, dtype=torch.long)

        perm = torch.randperm(len(self.y))
        self.x = self.x[perm]
        self.y = self.y[perm]

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

class ConceptImageDataset(Dataset):
    # returns image tensors only (no labels), as expected by Concept data iterators
    def __init__(self, kind='striped', n=200):
        images = []
        for i in range(n):
            if kind == 'striped':
                if i % 2 == 0:
                    img = make_horizontal_stripes()
                else:
                    img = make_vertical_stripes()
            elif kind == 'random':
                img = make_random_noise()
            else:
                raise ValueError(f'Unknown concept kind: {kind}')
            images.append(to_3ch_tensor(img))
        self.x = torch.stack(images)

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        return self.x[idx]

train_ds = StripeClassificationDataset(n_per_class=1000)
test_ds = StripeClassificationDataset(n_per_class=250)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

print('Train size:', len(train_ds), 'Test size:', len(test_ds))


In [ ]:
# quick visualization
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i in range(8):
    x, y = train_ds[i]
    ax = axes[i // 4, i % 4]
    ax.imshow(x[0].numpy(), cmap='gray', vmin=0.0, vmax=1.0)
    ax.set_title('horizontal' if y.item() == 0 else 'vertical')
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# model definition
class SmallStripeCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * (IMG_SIZE // 4) * (IMG_SIZE // 4), 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = SmallStripeCNN().to(DEVICE)
print(model)


In [ ]:
# model training
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred = model(xb).argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += yb.numel()
    return correct / max(total, 1)

EPOCHS = 6
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)

    train_loss = running_loss / len(train_ds)
    test_acc = evaluate(model, test_loader)
    print(f'Epoch {epoch:02d} | train_loss={train_loss:.4f} | test_acc={test_acc:.4f}')

print('Final test accuracy:', evaluate(model, test_loader))


In [ ]:
# concept creation
class DeviceDataLoader:
    # Wrap a DataLoader so each yielded batch is moved to the model device.
    def __init__(self, loader, device):
        self.loader = loader
        self.device = device

    def __iter__(self):
        for batch in self.loader:
            if isinstance(batch, (list, tuple)):
                if len(batch) == 1:
                    yield batch[0].to(self.device)
                else:
                    yield tuple(x.to(self.device) for x in batch)
            else:
                yield batch.to(self.device)

    def __len__(self):
        return len(self.loader)

striped_concept_loader = DataLoader(ConceptImageDataset(kind='striped', n=300), batch_size=32, shuffle=True)
random_concept_loader = DataLoader(ConceptImageDataset(kind='random', n=300), batch_size=32, shuffle=True)

striped_concept = Concept(id=0, name='striped', data_iter=DeviceDataLoader(striped_concept_loader, DEVICE))
random_concept = Concept(id=1, name='random', data_iter=DeviceDataLoader(random_concept_loader, DEVICE))

experimental_sets = [[striped_concept, random_concept]]

print('Concepts prepared:', [c.name for c in experimental_sets[0]])


In [ ]:
# TCAV computation
model.eval()

# Use an intermediate CNN layer
target_layer = 'features.3'

linear_cls = DefaultClassifier()
tcav = TCAV(
    model=model,
    layers=[target_layer],
    classifier=linear_cls,
    save_path='./tcav_cavs'
)

# Build CAVs (and keep the returned metadata for logging)
cav_meta = tcav.compute_cavs(experimental_sets=experimental_sets, force_train=True)
print('CAV metadata type:', type(cav_meta))
print('CAV metadata:', cav_meta)
print('Classifier used:', type(linear_cls).__name__)

# Pull one test batch for interpretation
x_test, y_test = next(iter(test_loader))
x_test = x_test.to(DEVICE)
y_test = y_test.to(DEVICE)

tcav_scores = tcav.interpret(
    inputs=x_test,
    experimental_sets=experimental_sets,
    target=y_test
)

print('Raw TCAV scores dictionary:')
print(tcav_scores)


In [ ]:
# results
def summarize_cav_accuracies(tcav_obj):
    print('\nCAV classifier accuracy (if available):')
    found = False
    if hasattr(tcav_obj, 'cavs'):
        for key, val in tcav_obj.cavs.items():
            # val can be nested dict or CAV object depending on Captum version
            if isinstance(val, dict):
                for k2, v2 in val.items():
                    stats = getattr(v2, 'stats', None)
                    if stats is not None:
                        print(f'  {key} / {k2}: {stats}')
                        found = True
            else:
                stats = getattr(val, 'stats', None)
                if stats is not None:
                    print(f'  {key}: {stats}')
                    found = True
    if not found:
        print('  Not exposed in tcav.cavs for this Captum version. See cav_meta above.')

def extract_metric(result_dict, layer, metric='sign_count'):
    exp_key = next(iter(result_dict.keys()))
    value = result_dict[exp_key][layer][metric]
    if torch.is_tensor(value):
        arr = value.detach().cpu().numpy()
    else:
        arr = np.asarray(value)
    return arr

sign_scores = extract_metric(tcav_scores, target_layer, metric='sign_count')
mag_scores = extract_metric(tcav_scores, target_layer, metric='magnitude')

print('\nTCAV sign_count scores shape:', sign_scores.shape)
print(sign_scores)
print('\nTCAV magnitude scores shape:', mag_scores.shape)
print(mag_scores)

summarize_cav_accuracies(tcav)


In [ ]:
# visualization + per-sample concept influence
concept_names = ['striped', 'random']

def per_sample_tcav_influence(img_tensor, predicted_label):
    img_tensor = img_tensor.unsqueeze(0).to(DEVICE)
    label_tensor = torch.tensor([predicted_label], device=DEVICE)
    score = tcav.interpret(inputs=img_tensor, experimental_sets=experimental_sets, target=label_tensor)
    s = extract_metric(score, target_layer, metric='sign_count').reshape(-1)
    return s

model.eval()
x_vis, y_vis = next(iter(test_loader))
x_vis = x_vis[:6]

with torch.no_grad():
    preds = model(x_vis.to(DEVICE)).argmax(dim=1).cpu().numpy()

fig, axes = plt.subplots(2, 3, figsize=(10, 6))
for i in range(6):
    infl = per_sample_tcav_influence(x_vis[i], int(preds[i]))
    top_idx = int(np.argmax(infl))

    ax = axes[i // 3, i % 3]
    ax.imshow(x_vis[i][0].numpy(), cmap='gray', vmin=0.0, vmax=1.0)
    ax.set_title(
        f'pred={"H" if preds[i]==0 else "V"} | top={concept_names[top_idx]}\n'
        f'striped={infl[0]:.3f}, random={infl[1]:.3f}'
    )
    ax.axis('off')

    print(f'Sample {i}: predicted={preds[i]}, concept influence -> striped={infl[0]:.4f}, random={infl[1]:.4f}, dominant={concept_names[top_idx]}')

plt.tight_layout()
plt.show()
